### 1. Import Library

In [21]:
### 1. Import Library

import warnings
import json
import time

from pathlib import Path

import joblib
import pandas as pd

from joblib import Parallel, delayed

from sklearn.cluster import KMeans
from sklearn.model_selection import ParameterGrid

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

### 2. Path Configuration

In [ ]:
### 2. Path Configuration

PREPROCESS_PATH = Path(
    r"C:\Users\PC\Documents\CADT_Y4\Internship_II\Project\AI_Credit_Scoring_System\model_development\unsupervised_learning\preprocess"
)

MODEL_PATH = Path(
    r"C:\Users\PC\Documents\CADT_Y4\Internship_II\Project\AI_Credit_Scoring_System\model_development\unsupervised_learning\models"
)

RESULT_PATH = Path(
    r"C:\Users\PC\Documents\CADT_Y4\Internship_II\Project\AI_Credit_Scoring_System\model_development\unsupervised_learning\results\kmeans"
)

MODEL_PATH.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

### 3. Load Dataset

In [23]:
### 3. Load Dataset

def load_dataset(prefix):

    X_train = pd.read_csv(
        PREPROCESS_PATH / f"{prefix}_X_train.csv"
    )

    X_test = pd.read_csv(
        PREPROCESS_PATH / f"{prefix}_X_test.csv"
    )

    return X_train, X_test

### 4. Hyperparameter Configuration

In [24]:
### 4. KMeans Hyperparameter Search

PARAM_GRID = {

    # Number of clusters
    "n_clusters": [2, 3, 4, 5, 6, 7],

    # Initialization method
    "init": [
        "k-means++",
        "random"
    ],

    # Number of centroid initializations
    "n_init": [10, 20, 30],

    # Maximum training iterations
    "max_iter": [300, 500]

}

# Speed up Silhouette Score during tuning
SILHOUETTE_SAMPLE_SIZE = 5000

### 5. Helper Function

In [25]:
def fit_and_score(params, X_train):
    """
    Train one KMeans model and compute its Silhouette Score.
    """

    model = KMeans(
        random_state=RANDOM_STATE,
        **params
    )

    labels = model.fit_predict(X_train)

    sample_size = (
        SILHOUETTE_SAMPLE_SIZE
        if len(X_train) > SILHOUETTE_SAMPLE_SIZE
        else None
    )

    score = silhouette_score(
        X_train,
        labels,
        sample_size=sample_size,
        random_state=RANDOM_STATE
    )

    return {
        "params": params,
        "score": score
    }

### 6. Training Function

In [26]:
def train_model(
    X_train,
    dataset_name,
    n_jobs=-1
):

    print("=" * 60)
    print(f"KMEANS TRAINING : {dataset_name}")
    print("=" * 60)

    start_time = time.time()

    parameter_list = list(
        ParameterGrid(PARAM_GRID)
    )

    print(
        f"Total Parameter Combinations : {len(parameter_list)}"
    )

    ############################################################
    # Hyperparameter Search
    ############################################################

    outputs = Parallel(
        n_jobs=n_jobs
    )(
        delayed(fit_and_score)(
            params,
            X_train
        )
        for params in parameter_list
    )

    ############################################################
    # Save Search Log
    ############################################################

    search_log = []

    for item in outputs:

        row = item["params"].copy()
        row["Silhouette"] = item["score"]
        search_log.append(row)

    search_log = pd.DataFrame(search_log)

    search_log.sort_values(
        "Silhouette",
        ascending=False,
        inplace=True
    )

    search_log.reset_index(
        drop=True,
        inplace=True
    )

    search_log.to_csv(

        RESULT_PATH /
        f"{dataset_name}_grid_search_log.csv",

        index=False

    )

    ############################################################
    # Best Parameter
    ############################################################

    best_result = max(
        outputs,
        key=lambda x: x["score"]
    )

    best_params = best_result["params"]
    best_score = best_result["score"]

    tuning_time = round(time.time() - start_time, 2)

    ############################################################
    # Save Best Parameters
    ############################################################

    with open(

        RESULT_PATH /
        f"{dataset_name}_best_params.json", "w"
    ) as f:

        json.dump(
            best_params,
            f,
            indent=4
        )

    ############################################################
    # Final Training
    ############################################################

    train_start = time.time()
    best_model = KMeans(
        random_state=RANDOM_STATE,
        **best_params

    )

    best_model.fit(
        X_train
    )

    training_time = round(time.time() - train_start, 4)

    ############################################################
    # Display
    ############################################################

    print("\nBest Parameters")
    print(best_params)
    print(f"\nBest Silhouette : {best_score:.4f}")
    print(f"Tuning Time     : {tuning_time:.2f} sec")
    print(f"Training Time   : {training_time:.4f} sec")
    print("=" * 60)

    return (
        best_model,
        tuning_time,
        training_time,
        best_score,
        best_params
    )

### 7. Evaluation Function

In [27]:
### 6. Evaluation Function

def evaluate_model(
    model,
    X_test,
    dataset_name
):

    labels = model.predict(X_test)

    results = {

        "Dataset": dataset_name,
        "Model": "KMeans",
        "Silhouette": round(
            silhouette_score(
                X_test,
                labels
            ),4
        ),

        "Davies_Bouldin": round(
            davies_bouldin_score(
                X_test,
                labels
            ), 4
        ),

        "Calinski_Harabasz": round(
            calinski_harabasz_score(
                X_test,
                labels
            ), 2
        )
    }

    return pd.DataFrame([results])

### 8. Train and Evaluation NEW Dataset

In [28]:
X_new_train, X_new_test = load_dataset("new")

(
    new_model,
    new_tuning_time,
    new_training_time,
    new_best_score,
    new_best_params
) = train_model(
    X_new_train,
    "NEW"
)

KMEANS TRAINING : NEW
Total Parameter Combinations : 72

Best Parameters
{'init': 'k-means++', 'max_iter': 300, 'n_clusters': 2, 'n_init': 10}

Best Silhouette : 0.1465
Tuning Time     : 54.29 sec
Training Time   : 0.3000 sec


In [29]:
results_new = evaluate_model(
    new_model,
    X_new_test,
    "NEW"
)

### 9. Train and Evaluation OLD Dataset 

In [30]:
X_old_train, X_old_test = load_dataset("old")

(
    old_model,
    old_tuning_time,
    old_training_time,
    old_best_score,
    old_best_params
) = train_model(
    X_old_train,
    "OLD"
)

KMEANS TRAINING : OLD
Total Parameter Combinations : 72

Best Parameters
{'init': 'k-means++', 'max_iter': 300, 'n_clusters': 2, 'n_init': 20}

Best Silhouette : 0.1218
Tuning Time     : 43.45 sec
Training Time   : 0.5782 sec


In [31]:
results_old = evaluate_model(
    old_model,
    X_old_test,
    "OLD"
)

### 10. Save Model

In [32]:
joblib.dump(
    new_model,
    MODEL_PATH / "NEW_KMEANS.pkl"
)

joblib.dump(
    old_model,
    MODEL_PATH / "OLD_KMEANS.pkl"
)

print("KMeans models saved")

KMeans models saved


### 11. Save Evaluation Results

In [33]:
results_all = pd.concat(
    [
        results_new,
        results_old
    ],
    ignore_index=True

)

results_all["Best_Silhouette"] = [
    round(new_best_score,4),
    round(old_best_score,4)
]

results_all["Tuning_Time_Seconds"] = [
    new_tuning_time,
    old_tuning_time
]

results_all["Training_Time_Seconds"] = [
    new_training_time,
    old_training_time

]

results_all.to_csv(
    RESULT_PATH /
    "kmeans_results.csv",
    index=False

)

print("\nEvaluation Results")
display(results_all)


Evaluation Results


,Dataset,Model,Silhouette,Davies_Bouldin,Calinski_Harabasz,Best_Silhouette,Tuning_Time_Seconds,Training_Time_Seconds
0,NEW,KMeans,0.1463,2.2746,1034.40,0.1465,54.29,0.3000
1,OLD,KMeans,0.1298,2.5533,777.42,0.1218,43.45,0.5782


### 12. Save Cluster Centers

In [34]:
pd.DataFrame(
    new_model.cluster_centers_,
    columns=X_new_train.columns
).to_csv(
    RESULT_PATH / "NEW_cluster_centers.csv",
    index=False
)

pd.DataFrame(
    old_model.cluster_centers_,
    columns=X_old_train.columns
).to_csv(
    RESULT_PATH / "OLD_cluster_centers.csv",
    index=False
)

### 13. Save Cluster Labels

In [35]:
new_cluster = X_new_train.copy()
new_cluster["Cluster"] = new_model.labels_

old_cluster = X_old_train.copy()
old_cluster["Cluster"] = old_model.labels_

new_cluster.to_csv(
    RESULT_PATH / "NEW_cluster_result.csv",
    index=False
)

old_cluster.to_csv(
    RESULT_PATH / "OLD_cluster_result.csv",
    index=False
)

### 14. Save Cluster Distribution

In [36]:
new_distribution = (
    new_cluster["Cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .reset_index(name="Count")
)

old_distribution = (
    old_cluster["Cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .reset_index(name="Count")
)

print("\nNEW USER")
display(new_distribution)

print("\nOLD USER")
display(old_distribution)

new_distribution.to_csv(
    RESULT_PATH /
    "NEW_cluster_distribution.csv",
    index=False
)

old_distribution.to_csv(
    RESULT_PATH /
    "OLD_cluster_distribution.csv",
    index=False
)


NEW USER


,Cluster,Count
0,0,20891
1,1,7109



OLD USER


,Cluster,Count
0,0,21306
1,1,6694


### 15. Save Cluster Means

In [37]:
new_means = (
    new_cluster
    .groupby("Cluster")
    .mean()
)

old_means = (
    old_cluster
    .groupby("Cluster")
    .mean()
)

display(
    new_means.round(2)
)

display(
    old_means.round(2)
)

new_means.to_csv(
    RESULT_PATH /
    "NEW_cluster_means.csv"
)

old_means.to_csv(
    RESULT_PATH /
    "OLD_cluster_means.csv"
)

,num__age,num__dependents,num__monthly_income,num__income_source_diversity,num__expense_ratio,num__monthly_outflow,num__net_cash_flow,num__loan_amount,num__loan_tenure,num__existing_loan_count,...,nom__loan_purpose_Repair of the House,nom__loan_purpose_Shopping,nom__loan_purpose_Startup,nom__loan_purpose_Travel,nom__loan_purpose_Vehicles,nom__loan_purpose_Wedding/birth anniversary,nom__asset_ownership_Car,nom__asset_ownership_Computer,nom__asset_ownership_Motobike,nom__asset_ownership_Phone
Cluster,,,,,,,,,,,,,,,,,,,,,
0,-0.03,0.47,-0.18,0.45,0.03,-0.13,-0.13,-0.12,0.04,0.01,...,0.1,0.1,0.1,0.1,0.1,0.1,0.25,0.25,0.26,0.25
1,0.12,0.56,1.41,0.45,-0.10,1.34,1.40,1.38,0.04,-0.01,...,0.1,0.1,0.1,0.1,0.1,0.1,0.24,0.26,0.25,0.25


,num__age,num__dependents,num__monthly_income,num__income_source_diversity,num__expense_ratio,num__monthly_outflow,num__net_cash_flow,num__loan_amount,num__loan_tenure,num__existing_loan_count,...,nom__loan_purpose_Repair of the House,nom__loan_purpose_Shopping,nom__loan_purpose_Startup,nom__loan_purpose_Travel,nom__loan_purpose_Vehicles,nom__loan_purpose_Wedding/birth anniversary,nom__asset_ownership_Car,nom__asset_ownership_Computer,nom__asset_ownership_Motobike,nom__asset_ownership_Phone
Cluster,,,,,,,,,,,,,,,,,,,,,
0,0.03,0.55,-0.17,0.44,0.04,-0.12,-0.12,-0.11,0.11,0.03,...,0.09,0.10,0.1,0.1,0.1,0.10,0.25,0.25,0.25,0.25
1,0.03,0.56,1.49,0.45,-0.11,1.40,1.47,1.40,0.12,-0.09,...,0.10,0.11,0.1,0.1,0.1,0.11,0.25,0.24,0.25,0.26
